In [9]:
import pandas as pd
import requests
import io
import time

def get_uniprot_sequence(uniprot_id):
    """Récupère la séquence FASTA d'UniProt pour un ID donné."""
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            # On ignore la première ligne (le header >sp|...)
            lines = response.text.splitlines()
            return "".join(lines[1:])
        else:
            print(f"Erreur pour {uniprot_id}: {response.status_code}")
            return None
    except Exception as e:
        print(f"Erreur réseau pour {uniprot_id}: {e}")
        return None

# Chargement de ton fichier de base
df = pd.read_csv('Dataset_gbm.csv')

# Récupération des séquences uniques pour gagner du temps
unique_ids = df['UniProt ID'].unique()
seq_map = {}

print(f"Téléchargement de {len(unique_ids)} séquences...")
count = 0
for uid in unique_ids:
    seq_map[uid] = get_uniprot_sequence(uid)
    time.sleep(0.1) # Pause courte pour respecter l'API
    count += 1
    if count % 50 == 0:
        print(count, " séquences téléchargées")


# On ajoute la colonne Sequence au dataset
df['Sequence'] = df['UniProt ID'].map(seq_map)

Téléchargement de 4250 séquences...
50  séquences téléchargées
100  séquences téléchargées
150  séquences téléchargées
200  séquences téléchargées
250  séquences téléchargées
300  séquences téléchargées
350  séquences téléchargées
400  séquences téléchargées
450  séquences téléchargées
500  séquences téléchargées
550  séquences téléchargées
600  séquences téléchargées
650  séquences téléchargées
700  séquences téléchargées
750  séquences téléchargées
800  séquences téléchargées
850  séquences téléchargées
900  séquences téléchargées
950  séquences téléchargées
1000  séquences téléchargées
1050  séquences téléchargées
1100  séquences téléchargées
1150  séquences téléchargées
1200  séquences téléchargées
1250  séquences téléchargées
1300  séquences téléchargées
1350  séquences téléchargées
1400  séquences téléchargées
1450  séquences téléchargées
1500  séquences téléchargées
1550  séquences téléchargées
1600  séquences téléchargées
1650  séquences téléchargées
1700  séquences téléchargée

In [13]:
import re

def parse_mutation(mutation_str):
    """Parse la mutation, pour accessibilité facile à WT, position, mutant"""
    match = re.match(r"([A-Z])(\d+)([A-Z])", mutation_str)
    if match:
        wt, pos, mut = match.groups()
        return wt, int(pos), mut
    return None, None, None


def parse_aaindex1(file_path):
    """Parse le fichier aaindex1 pour extraire les indices numériques."""
    aaindex_db = {}
    # Ordre standard des AA dans la section 'I' de AAindex1
    aa_order = ['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V']
    
    with open(file_path, 'r') as f:
        # On sépare chaque entrée par le délimiteur '//'
        entries = f.read().split('//')
        for entry in entries:
            if not entry.strip(): continue
            
            # Récupérer l'ID de la propriété (ligne H)
            header = re.search(r'^H\s+(\S+)', entry, re.M)
            # Récupérer les valeurs numériques (section I)
            values_match = re.search(r'^I\s+.*?\n\s+(.*)', entry, re.S | re.M)
            
            if header and values_match:
                prop_id = header.group(1)
                # Nettoyer et extraire les 20 nombres
                raw_vals = re.findall(r'-?\d+\.\d+|-?\d+', values_match.group(1))
                if len(raw_vals) == 20:
                    aaindex_db[prop_id] = {aa: float(v) for aa, v in zip(aa_order, raw_vals)}
    return aaindex_db

# Chargement de la base
db = parse_aaindex1('aaindex1')

# Update du df des mutations
df[['Wildtype', 'Position', 'Mutant']] = df['Mutation'].apply(lambda x: pd.Series(parse_mutation(x)))

In [15]:
# On récupère déja ce fichier avec mutations, séquence et parsing de la mutation
df.to_csv("mutation_parse_sequence.csv")

In [18]:
def extract_features_for_mutation(row, db, prop_list, window_size=6):
    res = {}
    seq = row['Sequence']
    
    # 1. Sécurité : Vérifier si la séquence existe
    if pd.isna(seq) or not seq:
        # On retourne des zéros pour ne pas bloquer le pipeline
        for p_id in prop_list:
            res[f"{p_id}_delta"] = 0.0
            res[f"{p_id}_window"] = 0.0
        return pd.Series(res)

    try:
        pos = int(row['Position']) - 1 # Index Python
    except (ValueError, TypeError):
        # Si la position n'est pas un nombre
        return pd.Series({f"{p_id}_{suffix}": 0.0 for p_id in prop_list for suffix in ['delta', 'window']})

    for p_id in prop_list:
        if p_id not in db: continue
        
        # Calcul du Delta P
        wt = row['Wildtype']
        mut = row['Mutant']
        val_wt = db[p_id].get(wt, 0)
        val_mut = db[p_id].get(mut, 0)
        res[f"{p_id}_delta"] = val_mut - val_wt
        
        # Calcul de la moyenne de fenêtre
        start = max(0, pos - window_size)
        end = min(len(seq), pos + window_size + 1)
        sub_seq = seq[start:end]
        
        # 2. Sécurité : Vérifier que la sous-séquence n'est pas vide
        if len(sub_seq) > 0:
            window_vals = [db[p_id].get(aa, 0) for aa in sub_seq]
            res[f"{p_id}_window"] = sum(window_vals) / len(window_vals)
        else:
            res[f"{p_id}_window"] = 0.0
            
    return pd.Series(res)


# Liste des IDs (Il faudrait idéalement les 268 IDs de la Table S7 de l'article) [cite: 67, 224]
# Ici, on prend un échantillon pour l'exemple
selected_props = ['ANDN920101', 'ARGP820101', 'ARGP820102'] 

# Application du pipeline
feature_df = df.apply(lambda row: extract_features_for_mutation(row, db, selected_props), axis=1)
final_dataset = pd.concat([df, feature_df], axis=1)

# Sauvegarde
#final_dataset.to_csv('gbm_with_features.csv', index=False)

In [32]:

#print(final_dataset.head(50))
x = final_dataset[(final_dataset["UniProt ID"]=="P12110") & (final_dataset["Mutation"]=="R366Q")]
print(x)

     UniProt ID   Gene Mutation   Class  \
7308     P12110  CO6A2    R366Q  Driver   

                                               Sequence Wildtype  Position  \
7308  MLQGTCSVLLLWGILGAIQAQQQEVISPDTTERNNNCPEKTDCPIH...        R       366   

     Mutant  ANDN920101_delta  ANDN920101_window  ARGP820101_delta  \
7308      Q             -0.01           4.253846              -0.6   

      ARGP820101_window  ARGP820102_delta  ARGP820102_window  
7308           0.433846              0.52                0.5  
